<a href="https://colab.research.google.com/github/Steve-Falkovsky/Hypencoder-Entity-Linking/blob/main/notebooks/Hypencoder_Vs_fine_tuned_Hypencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Environment lock (run once per fresh runtime) ---
# because libraries like transformers can change and make your code non-runnable :)
# Torch updates also broke the code but installing it in colab to an older version
# takes 2 minutes every time so I just solved the one bug

!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/env_setup.py
import env_setup
env_setup.install_env()

This should take 20-30 seconds
Installing transformers==4.50.0 and sentence_transformers==2.2.2...
Installed. Restarting runtime (session) is required.


In [1]:
import os

REPO_NAME = "Hypencoder-Entity-Linking"
GIT_URL = f"https://github.com/Steve-Falkovsky/{REPO_NAME}.git"
MAIN_BRANCH_NAME = "main"
OLD_PYTORCH_BRANCH_NAME = "oldtorch"

!git clone -b {MAIN_BRANCH_NAME} --single-branch {GIT_URL}

# Move into the downloaded repo (The Root)
os.chdir(REPO_NAME)

%pip install -q -e "./hypencoder-paper"

os.chdir("hypencoder-paper")

print(f"📍 Working Directory is now: {os.getcwd()}")
print("✅ Environment Ready!")

Cloning into 'Hypencoder-Entity-Linking'...
remote: Enumerating objects: 610, done.
remote: Counting objects: 100% (243/243), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 610 (delta 164), reused 132 (delta 73), pack-reused 367 (from 1)
Receiving objects: 100% (610/610), 1.08 MiB | 14.78 MiB/s, done.
Resolving deltas: 100% (355/355), done.
  Preparing metadata (setup.py) ... done
📍 Working Directory is now: /content/Hypencoder-Entity-Linking/hypencoder-paper
✅ Environment Ready!


In [2]:
from datasets import load_dataset

# there are all "positive" pairs
dataset = load_dataset("Stevenf232/BC5CDR_MeSH2015_nameonly")

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

name_only_train.jsonl: 0.00B [00:00, ?B/s]

name_only_val.jsonl: 0.00B [00:00, ?B/s]

name_only_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2654 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2559 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2656 [00:00<?, ? examples/s]

In [3]:
data_pairs = dataset['test']
print(data_pairs)

mention_names = data_pairs['mention']
entity_names = data_pairs['entity']
print(mention_names[:3])
print(entity_names[:3])

Dataset({
    features: ['mention', 'entity', 'id'],
    num_rows: 2656
})
['oral ulcers', 'GSSG', 'blood loss']
['Oral Ulcer', 'Glutathione Disulfide', 'Postpartum Hemorrhage']


### Load the model

In [4]:
# Core Hypencoder model for outputing dense vector representations
from hypencoder_cb.modeling.hypencoder import Hypencoder, HypencoderDualEncoder, TextEncoder
from transformers import AutoTokenizer

model_name = "Stevenf232/hypencoder_2layer_SapBERT_nameOnly"


dual_encoder = HypencoderDualEncoder.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


query_encoder: Hypencoder = dual_encoder.query_encoder
passage_encoder: TextEncoder = dual_encoder.passage_encoder

config.json:   0%|          | 0.00/927 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/483M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

### Move the model to the GPU

In [5]:
import torch

# Setup the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # This should say 'cuda'

# Move the model to the GPU
passage_encoder.to(device)
query_encoder.to(device)

Using device: cuda


Hypencoder(
  (transformer): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, element

### Load datasets and tokenise

In [6]:
# convert from type "datasets" to python list
queries = list(mention_names)
passages = list(entity_names)


# the output of the tokenizer contains 3 fields:
# input_ids, token_type_ids, and attention_mask
# all contain a tensor in the shape (number of queries, max number of tokens)

query_inputs = tokenizer(queries, return_tensors="pt", padding=True, truncation=True)
passage_inputs = tokenizer(passages, return_tensors="pt", padding=True, truncation=True)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [7]:
print(f"query_inputs:\n{query_inputs}")
print("\n\n\n")
print(f"passage_inputs:\n{passage_inputs}")

query_inputs:
{'input_ids': tensor([[    2,  4726, 16225,  ...,     0,     0,     0],
        [    2, 24782,     3,  ...,     0,     0,     0],
        [    2,  2877,  3257,  ...,     0,     0,     0],
        ...,
        [    2,  5761,  1920,  ...,     0,     0,     0],
        [    2, 13096,  1036,  ...,     0,     0,     0],
        [    2, 10205, 12013,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}




passage_inputs:
{'input_ids': tensor([[    2,  4726, 10096,  ...,     0,     0,     0],
        [    2,  9153, 11830,  ...,     0, 

# Passage Encodings


In [8]:
from tqdm import tqdm
import torch
from torch.amp import autocast

def batch_encode_passages(encoder ,passages):
  batch_size=256
  entity_name_features = []

  num_passages = passages["input_ids"].shape[0]

  with torch.no_grad(): # Disable gradient calculation (saves tons of memory)
    for i in tqdm(range(0, num_passages, batch_size), desc="Extracting features"):

        # extract entity features
        # Autocast does the math in fp16 where possible (default is fp32)
        # this will save memory and increase speed. The loss in precision shouldn't matter much (can check on a small sample if we want)
        with autocast("cuda"):
          features = encoder(
              input_ids=passages["input_ids"][i:i + batch_size].to(device),
              attention_mask=passages["attention_mask"][i:i + batch_size].to(device)
            ).representation

          entity_name_features.append(features.detach().cpu()) # Detach and move to CPU to save VRAM/RAM


  features_tensor = torch.cat(entity_name_features, dim=0)

  return features_tensor

In [9]:
passage_embeddings = batch_encode_passages(passage_encoder, passage_inputs)

Extracting features: 100%|██████████| 11/11 [00:01<00:00,  7.35it/s]


##Now, we create the q-nets.

For each q-net, we feed through it all the passages the calculate the similarity.

But the q-nets are created in batches, and every batch is represented as a single object `NoTorchSequential`. (Check out the `RepeatedDenseBlockConverter` class in q_net.py for more info)

This object expects an input in the shape (N, M, H):

* N = number of queries (mentions)

* M = number of passages (entities)

* H = Hidden dimension (e.g., 768 for BERT)



---



The passage embeddings have the shape (M, H) so we must create an additional dimension of size N.

This will be done like so:
`passages_batch = passages.unsqueeze(0).expand(num_queries, -1, -1)`

* `.unsqueeze()` adds a new dimension (in our case at location 0)

* `.expand()` "expands" that new dimension to be size "num_queries"

* `.expand()` creates a view, so it costs almost 0 memory! (compared to .repeat() which changes the tensor)

# Q-nets take **a lot** of memory.

Instead of creating all of them and then doing the similarity calculation, we will create batches and calculate similarities for just those q-nets, then discard those q-nets and move on to the next batch.

In [10]:
def batch_encode_queries(encoder, queries, passage_embeddings):
  batch_size = 8
  similarity_scores = []

  num_queries = queries["input_ids"].shape[0]

  with torch.no_grad():
    for i in tqdm(range(0, num_queries, batch_size), desc="Creating q-nets and calculating similarity scores"):

        # create q-nets
        with autocast("cuda"):
          q_nets = encoder(
              input_ids=queries["input_ids"][i:i + batch_size].to(device),
              attention_mask=queries["attention_mask"][i:i + batch_size].to(device)
            ).representation


        passages_gpu = passage_embeddings.to(device)

        # Note: we use q_nets.num_queries (our repo's noTorch equivalent of q_nets.shape[0]) instead of batch_size
        # because the total number might not be divisible by batch_size so the last batch might be smaller than the actual batch size
        passages_batch = passages_gpu.unsqueeze(0).expand(q_nets.num_queries, -1, -1)

        # calculate similarity
        batch_scores = q_nets(passages_batch)
        similarity_scores.append(batch_scores.detach().cpu())


  scores_tensor = torch.cat(similarity_scores, dim=0)
  return scores_tensor


In [11]:
similarity_scores = batch_encode_queries(query_encoder, query_inputs, passage_embeddings)

Creating q-nets and calculating similarity scores: 100%|██████████| 332/332 [00:12<00:00, 27.02it/s]


In [12]:
similarity_scores

tensor([[[13.6013],
         [-0.2032],
         [-0.3839],
         ...,
         [ 1.7606],
         [ 1.0936],
         [ 0.7954]],

        [[ 2.1349],
         [14.1166],
         [ 0.7307],
         ...,
         [-1.6025],
         [-1.6141],
         [-2.6713]],

        [[ 4.1981],
         [-2.8584],
         [ 9.9953],
         ...,
         [-1.8864],
         [-0.4400],
         [ 6.3838]],

        ...,

        [[ 2.9150],
         [ 0.1793],
         [ 1.7179],
         ...,
         [18.9699],
         [ 4.4332],
         [ 3.7632]],

        [[ 3.5086],
         [ 2.5860],
         [-2.9924],
         ...,
         [ 3.2958],
         [16.6911],
         [ 2.0982]],

        [[-0.0696],
         [-1.2695],
         [ 5.0148],
         ...,
         [ 1.8090],
         [ 1.4769],
         [18.8361]]])

In [13]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def evaluate(data_pairs):
  correct_count = 0
  top_idxs = torch.argmax(similarity_scores,dim=1).flatten()

  for i in range(len(queries)):
      top_idx = top_idxs[i]
      # the conversion to int from here on out is because the original idx is of type numpy.int64
      top_match_id = data_pairs["id"][int(top_idx)]
      correct_id = data_pairs["id"][int(i)]

      if top_match_id == correct_id:
          correct_count += 1

      mention_name = data_pairs["mention"][int(i)]
      top_match = data_pairs["entity"][int(top_idx)]
      correct_name = data_pairs["entity"][int(i)]
      print(f"mention_name: {mention_name}\ncorrect entity name: {correct_name}\ntop_match: {top_match}\n")


  print(f"total comparisons: {len(queries)}")
  print(f"correct comparisons: {correct_count}")
  print(f"accuracy: {correct_count / len(queries)}")

In [14]:
evaluate(data_pairs)

Streaming output truncated to the last 5000 lines.

mention_name: Atrial thrombosis
correct entity name: Coronary Thrombosis
top_match: Atrial Fibrillation

mention_name: cognitive disorders
correct entity name: Cognition Disorders
top_match: Cognition Disorders

mention_name: CYP
correct entity name: Cyclophosphamide
top_match: Picrotoxin

mention_name: ZnSO(4)
correct entity name: Zinc Sulfate
top_match: Status Epilepticus

mention_name: phenylbutazone
correct entity name: Phenylbutazone
top_match: Phenylbutazone

mention_name: glioblastoma
correct entity name: Glioblastoma
top_match: Glioblastoma

mention_name: toxocity
correct entity name: Drug-Related Side Effects and Adverse Reactions
top_match: Poisoning

mention_name: albuterol
correct entity name: Albuterol
top_match: Albuterol

mention_name: neuropathy
correct entity name: Nervous System Diseases
top_match: Polyneuropathies

mention_name: tetrabenazine
correct entity name: Tetrabenazine
top_match: Tetrabenazine

mention_name:

Save results to file:

In [15]:
import contextlib
from pathlib import Path

dir = "results"
file = "2layersNameOnly.txt"
path = os.path.join(dir, file)

# Creates the folder if it's missing; does nothing if it already exists
Path(dir).mkdir(parents=True, exist_ok=True)

with open(path, 'w') as f:
    with contextlib.redirect_stdout(f):
        evaluate(data_pairs)
